In [ ]:
import json
import math
import os
import itertools
import importlib
import shutil
import subprocess
import traceback
from datetime import datetime, timezone
from pathlib import Path

import h5py
import mne
import numpy as np
import pandas as pd
import scipy.io
import scipy.integrate

In [ ]:
ROOT = str(Path.home() / "fnirs-representation-learning")
DATASET_DIRNAME = "snirf_dataset_2"
OUTPUT_DIRNAME = "outputs_benchmark_v7"
OUTPUT_PREFIX = "benchmark_v7"

# Leave this empty to run every Subj* folder in the dataset directory.
SUBJECT_NAMES = ["Subj94", "Subj100"]

SHORT_SEPARATION_THRESHOLD_M = 0.015
LONG_SEPARATION_THRESHOLD_M = 0.025
LOCAL_SS_MAX_DISTANCE_M = 0.015
MULTI_SS_K = 3
POOLED_SS_N_COMPONENTS = 2
SS_AUX_N_COMPONENTS = 3

STRICT_SCI_THRESHOLD = 0.50
LOOSE_SCI_THRESHOLD = 0.35
STRICT_SNR_THRESHOLD = 2.0
STRICT_NEGATIVE_FRACTION_THRESHOLD = 0.001

FILTER_LOW_HZ = 0.01
FILTER_HIGH_HZ = 0.20
FILTER_HIGHPASS_ONLY_HZ = 0.01
PPF_VALUE = 0.1
STIM_DURATION_S = 1.0
DRIFT_HIGH_PASS_HZ = 0.01
WAVELET_IQR_MULTIPLIER = 1.5
WAVELET_NAME = "db2"
WAVELET_PADDING_MODE = "periodization"
DLPFC_FRONTAL_QUANTILE = 0.65
DLPFC_LATERAL_QUANTILE = 0.50

EPOCH_TMIN = -5.0
EPOCH_TMAX = 30.0
BASELINE_WINDOW = (-5.0, 0.0)
RESPONSE_WINDOW = (4.0, 8.0)

FIR_RESAMPLE_SFREQ_HZ = 1.0
FIR_DELAYS_SCANS = list(range(0, 26))

EMPIRICAL_NULL_SHIFT_COUNT = 50
EMPIRICAL_NULL_MIN_SHIFT_S = 20.0

OVERWRITE = False
WRITE_CSV = True
WRITE_PARQUET = True

# Set this to False if you want to skip MATLAB/AnalyzIR pipelines.
USE_MATLAB = True
USE_MATLAB_ENGINE = True
PREFER_MATLAB_ENGINE = True
MATLAB_CMD = "matlab"
MATLAB_TIMEOUT_S = 7200
MATLAB_STARTUP_OPTIONS = ""
ANALYZIR_PATH = None

TRUTH_TEMPLATE_DIR = "/home/asunkari/fnirs-representation-learning/synthetic_hrf_generation"
TRUTH_TEMPLATE_MAP = {
    "hrf_20": "hrf_20.mat",
    "hrf_50": "hrf_50.mat",
    "hrf_100": "hrf_100.mat",
}

In [ ]:
FILE_SPECS = [
    {
        "label": "no_hrf",
        "filename": "resting_clean.snirf",
        "amplitude_value": 0,
        "is_null": True,
        "annotation_source_filename": "resting_hrf_20.snirf",
    },
    {
        "label": "hrf_20",
        "filename": "resting_hrf_20.snirf",
        "amplitude_value": 20,
        "is_null": False,
        "annotation_source_filename": None,
    },
    {
        "label": "hrf_50",
        "filename": "resting_hrf_50.snirf",
        "amplitude_value": 50,
        "is_null": False,
        "annotation_source_filename": None,
    },
    {
        "label": "hrf_100",
        "filename": "resting_hrf_100.snirf",
        "amplitude_value": 100,
        "is_null": False,
        "annotation_source_filename": None,
    },
]

In [ ]:
PIPELINE_SPECS = [
    {
        "label": "NoSS_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "none",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> NoSS -> Glover -> AUTO",
    },
    {
        "label": "LocalSS_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> Glover -> AUTO",
    },
    {
        "label": "PooledPCA2_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "pooled_pca2",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> PooledPCA2 -> Glover -> AUTO",
    },
    {
        "label": "SSAuxPCA_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "ss_aux_pca",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> SSAuxPCA -> Glover -> AUTO",
    },
    {
        "label": "MultiSSOrth3_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "multi_ss_orth3",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> MultiSSOrth3 -> Glover -> AUTO",
    },
    {
        "label": "NoSS_Glover_OLS",
        "backend": "python",
        "nuisance_method": "none",
        "hrf_model": "glover",
        "solver": "ols",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> NoSS -> Glover -> OLS",
    },
    {
        "label": "NoSS_Glover_ARIRLS",
        "backend": "matlab_arirls",
        "nuisance_method": "none",
        "hrf_model": "glover",
        "solver": "arirls",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> NoSS -> Glover -> AR-IRLS",
    },
    {
        "label": "LocalSS_Glover_OLS",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "ols",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> Glover -> OLS",
    },
    {
        "label": "LocalSS_Glover_ARIRLS",
        "backend": "matlab_arirls",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "arirls",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> Glover -> AR-IRLS",
    },
    {
        "label": "LocalSS_SPM_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "spm",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> SPM -> AUTO",
    },
    {
        "label": "LocalSS_SPM_OLS",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "spm",
        "solver": "ols",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> SPM -> OLS",
    },
    {
        "label": "LocalSS_FIR_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "fir",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> FIR -> AUTO",
    },
    {
        "label": "LocalSS_Gamma_ARIRLS",
        "backend": "matlab_arirls",
        "nuisance_method": "local_nearest",
        "hrf_model": "gamma",
        "solver": "arirls",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": False,
        "comparison_group": "core",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> Gamma -> AR-IRLS",
    },
    {
        "label": "LooseQC_LocalSS_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "loose_sci",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": True,
        "comparison_group": "secondary",
        "description": "LooseQC -> TDDR -> band-pass -> LocalSS -> Glover -> AUTO",
    },
    {
        "label": "WaveletMC_LocalSS_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "wavelet",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": True,
        "comparison_group": "secondary",
        "description": "StrictQC -> WaveletMC -> band-pass -> LocalSS -> Glover -> AUTO",
    },
    {
        "label": "HighPassOnly_LocalSS_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "highpass_only",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": True,
        "comparison_group": "secondary",
        "description": "StrictQC -> TDDR -> high-pass-only -> LocalSS -> Glover -> AUTO",
    },
    {
        "label": "NoMotion_LocalSS_Glover_AUTO",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "glover",
        "solver": "auto",
        "pruning_style": "strict_combined",
        "motion_method": "none",
        "filter_mode": "bandpass",
        "use_block_average": False,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": True,
        "comparison_group": "secondary",
        "description": "StrictQC -> no motion correction -> band-pass -> LocalSS -> Glover -> AUTO",
    },
    {
        "label": "BlockAvg_LocalSS",
        "backend": "python",
        "nuisance_method": "local_nearest",
        "hrf_model": "block_average",
        "solver": "none",
        "pruning_style": "strict_combined",
        "motion_method": "tddr",
        "filter_mode": "bandpass",
        "use_block_average": True,
        "include_in_empirical_null": True,
        "include_in_primary_variability": True,
        "secondary_pipeline": True,
        "comparison_group": "secondary",
        "description": "StrictQC -> TDDR -> band-pass -> LocalSS -> block average",
    },
]


def root_path(config=None):
    root = ROOT if config is None else config["root"]
    return Path(root).expanduser().resolve()


def dataset_path(config=None):
    if config is None:
        return root_path() / DATASET_DIRNAME
    return root_path(config) / config["dataset_dirname"]


def output_path(config=None):
    if config is None:
        return root_path() / OUTPUT_DIRNAME
    return root_path(config) / config["output_dirname"]


def jobs_path(config=None):
    return output_path(config) / "job_results"


def aggregate_path(config=None):
    return output_path(config) / "aggregate"


def config_snapshot():
    return {
        "root": ROOT,
        "dataset_dirname": DATASET_DIRNAME,
        "output_dirname": OUTPUT_DIRNAME,
        "output_prefix": OUTPUT_PREFIX,
        "subject_names": list(SUBJECT_NAMES),
        "file_specs": [dict(x) for x in FILE_SPECS],
        "pipeline_specs": [dict(x) for x in PIPELINE_SPECS],
        "short_separation_threshold_m": SHORT_SEPARATION_THRESHOLD_M,
        "long_separation_threshold_m": LONG_SEPARATION_THRESHOLD_M,
        "local_ss_max_distance_m": LOCAL_SS_MAX_DISTANCE_M,
        "multi_ss_k": MULTI_SS_K,
        "pooled_ss_n_components": POOLED_SS_N_COMPONENTS,
        "ss_aux_n_components": SS_AUX_N_COMPONENTS,
        "strict_sci_threshold": STRICT_SCI_THRESHOLD,
        "loose_sci_threshold": LOOSE_SCI_THRESHOLD,
        "strict_snr_threshold": STRICT_SNR_THRESHOLD,
        "strict_negative_fraction_threshold": STRICT_NEGATIVE_FRACTION_THRESHOLD,
        "filter_low_hz": FILTER_LOW_HZ,
        "filter_high_hz": FILTER_HIGH_HZ,
        "filter_highpass_only_hz": FILTER_HIGHPASS_ONLY_HZ,
        "ppf_value": PPF_VALUE,
        "stim_duration_s": STIM_DURATION_S,
        "drift_high_pass_hz": DRIFT_HIGH_PASS_HZ,
        "wavelet_iqr_multiplier": WAVELET_IQR_MULTIPLIER,
        "wavelet_name": WAVELET_NAME,
        "wavelet_padding_mode": WAVELET_PADDING_MODE,
        "dlpfc_frontal_quantile": DLPFC_FRONTAL_QUANTILE,
        "dlpfc_lateral_quantile": DLPFC_LATERAL_QUANTILE,
        "epoch_tmin": EPOCH_TMIN,
        "epoch_tmax": EPOCH_TMAX,
        "baseline_window": BASELINE_WINDOW,
        "response_window": RESPONSE_WINDOW,
        "fir_resample_sfreq_hz": FIR_RESAMPLE_SFREQ_HZ,
        "fir_delays_scans": list(FIR_DELAYS_SCANS),
        "empirical_null_shift_count": EMPIRICAL_NULL_SHIFT_COUNT,
        "empirical_null_min_shift_s": EMPIRICAL_NULL_MIN_SHIFT_S,
        "overwrite": OVERWRITE,
        "write_csv": WRITE_CSV,
        "write_parquet": WRITE_PARQUET,
        "use_matlab": USE_MATLAB,
        "use_matlab_engine": USE_MATLAB_ENGINE,
        "prefer_matlab_engine": PREFER_MATLAB_ENGINE,
        "matlab_cmd": MATLAB_CMD,
        "matlab_timeout_s": MATLAB_TIMEOUT_S,
        "matlab_startup_options": MATLAB_STARTUP_OPTIONS,
        "analyzir_path": ANALYZIR_PATH,
        "truth_template_dir": TRUTH_TEMPLATE_DIR,
        "truth_template_map": dict(TRUTH_TEMPLATE_MAP),
    }